# DataFrame statistical distributions

**Workflow 3 — analyse music: statistics and pattern search.** This notebook starts from one common-notation MEI file, builds `df_pitch` / `df_events`, and walks through CAMAT's DataFrame distribution helpers: pitch, duration, pitch class, successive transitions, melodic intervals, and onset position within the measure.

It is self-contained: it does **not** require [`mei_parse_tables.ipynb`](mei_parse_tables.ipynb) or [`mei_annotate_selection.ipynb`](mei_annotate_selection.ipynb) to have been run. Binary matrices and pattern search are in [`binary_roundtrip.ipynb`](binary_roundtrip.ipynb) and [`binary_pattern_search.ipynb`](binary_pattern_search.ipynb).

Common-notation MEI (MEI 5.1 CMN) only; see [Known limitations](../docs/known-limitations.md). Companion guide: [Statistics](../docs/guides/statistics.md).

**What you do**

1. Keep the Bach sample URL, or set `MEI_SOURCE` to a local `.mei` file.
2. If using the URL and it is not cached yet, set `RUN_FETCH = True` once.
3. Parse once, optionally keep a short selection, then run each distribution section.


In [ ]:
# Cloud Jupyter: install CAMAT and copy notebooks/test_corpus if they are not already here.
try:
    import setup_camat
except ModuleNotFoundError:
    pass
try:
    from camat.notebook_workspace import prepare_notebook
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "camat"])
    from camat.notebook_workspace import prepare_notebook

prepare_notebook()

# You can leave this cell unchanged.
try:
    from camat import (
        check_monophonic_input,
        display_duration_distribution,
        display_filtered_piano_roll,
        display_melodic_interval_distribution,
        display_onset_position_histogram,
        display_pitch_class_distributions,
        display_pitch_distribution,
        display_successive_pitch_transition_heatmaps,
        find_camat_root,
        parse_files,
        resolve_mei_source,
    )
except ModuleNotFoundError:
    import setup_camat
    from camat import (
        check_monophonic_input,
        display_duration_distribution,
        display_filtered_piano_roll,
        display_melodic_interval_distribution,
        display_onset_position_histogram,
        display_pitch_class_distributions,
        display_pitch_distribution,
        display_successive_pitch_transition_heatmaps,
        find_camat_root,
        parse_files,
        resolve_mei_source,
    )

from IPython.display import display


## 1. Point at one MEI source

Same Bach *Ein feste Burg* sample as Workflow 2. Downloads stay off until `RUN_FETCH = True`. `PLOTTING_BACKEND` is shared by every plot below (`plt` or `bokeh`).


In [ ]:
ROOT = find_camat_root()

BACH_SAMPLE_URL = (
    "https://raw.githubusercontent.com/music-encoding/sample-encodings/"
    "main/MEI_5.0/Music/Complete_examples/Bach-JS_Ein_feste_Burg.mei"
)

# Default: remote CMN sample (cached on first fetch).
MEI_SOURCE = BACH_SAMPLE_URL
# Local alternative (no network):
# MEI_SOURCE = "path/to/your_score.mei"

# Downloads/cache writes stay off until this is True (URL sources only).
RUN_FETCH = False

# Shared by every distribution plot in this notebook.
PLOTTING_BACKEND = "bokeh"  # "plt" | "bokeh"


## 2. Resolve and parse

`resolve_mei_source(..., fetch=RUN_FETCH, shared_cache=True)` uses the same download cache as `parse_files`. `parse_enharmonic=True` is required for enharmonic pitch and pitch-class views.


In [ ]:
mei_path = resolve_mei_source(
    MEI_SOURCE,
    fetch=RUN_FETCH,
    shared_cache=True,
    repo_root=ROOT,
)

if mei_path is None:
    print("Skipped. Remote MEI is not cached yet.")
    print("Set RUN_FETCH = True once to download, or point MEI_SOURCE at a local .mei file.")
    results = None
    dfs_by_name = None
    df_pitch = None
    df_events = None
else:
    results, dfs_by_name, _last_df = parse_files(
        [str(mei_path)],
        backend="none",
        display_preview_df_pitch=False,
        display_preview_df_events=False,
        include_xml_ids=True,
        parse_enharmonic=True,
        quiet_native_warnings=True,
        use_remote_cache=True,
    )
    df_pitch = results[0]["df_pitch"]
    df_events = results[0]["df_events"]
    print(f"Source:     {results[0].get('source', mei_path)}")
    print(f"df_pitch:   {len(df_pitch)} rows")
    print(f"df_events:  {len(df_events)} rows")
    print(f"Voices:     {sorted(df_pitch['Voice'].dropna().unique().tolist()) if 'Voice' in df_pitch.columns else []}")
    display(df_pitch.head(8))


## 3. Optional selection (one score only)

Keep multi-source comparison simple: the full `df_pitch`, or full score plus one filtered selection. This is not a multi-file corpus demo.


In [ ]:
# Compare full score against a short excerpt (set False to analyse only df_pitch).
COMPARE_SELECTION = True
MEASURE_RANGE = (1, 4)           # inclusive, or None
VOICE_QUERY = ["P1 - Voice 1"]   # exact names, or None for all voices

if df_pitch is None:
    print("Skipped. Resolve and parse a source first.")
    selection = None
    analysis_sources = None
    analysis_labels = None
elif not COMPARE_SELECTION:
    selection = None
    analysis_sources = df_pitch
    analysis_labels = ["full score"]
    print("Using full df_pitch only.")
else:
    selection = display_filtered_piano_roll(
        df_pitch,
        measure_range=MEASURE_RANGE,
        voice_query=VOICE_QUERY,
        results=results,
        plotting_backend=PLOTTING_BACKEND,
        show_measure_lines=True,
        colorize_voices=True,
        display_selection=True,
        display_mode="head",
        display_max_rows=10,
    )
    analysis_sources = (selection, df_pitch)
    analysis_labels = ["selection", "full score"]
    print(f"Selection rows: {len(selection)}")


## 4. Pitch distribution

`display_pitch_distribution` counts written or sounding pitches. With `parse_enharmonic=True`, `pitch_axis="pitch enharmonic"` uses the `Pitch Enharmonic` column.


In [ ]:
PITCH_X_AXIS = "pitch enharmonic"  # "midi" | "pitch real" | "pitch enharmonic"
ORDER_X_AXIS_BY = "midi"           # sort categories by MIDI height
NORMALIZE = True

if analysis_sources is None:
    print("Skipped. Parse a score first.")
    pitch_counts = None
else:
    pitch_counts = display_pitch_distribution(
        analysis_sources,
        source_labels=analysis_labels,
        pitch_axis=PITCH_X_AXIS,
        order_axis_by=ORDER_X_AXIS_BY,
        backend=PLOTTING_BACKEND,
        plot_width=1000,
        plot_height=350,
        show_hover=True,
        show_table=True,
        normalize=NORMALIZE,
        float_format=".3f",
    )


## 5. Duration distribution

Pick one explicit duration column. Segment duration (`Duration`) and tied logical duration (`Logical Duration`) answer different questions — see [`duration_semantics_examples.ipynb`](duration_semantics_examples.ipynb).


In [ ]:
DURATION_COLUMN = "Duration"  # "Duration" | "Logical Duration"
DROP_ZERO_DURATIONS = True
DURATION_NORMALIZE = True

if analysis_sources is None:
    print("Skipped. Parse a score first.")
    duration_counts = None
else:
    duration_counts = display_duration_distribution(
        analysis_sources,
        source_labels=analysis_labels,
        duration_column=DURATION_COLUMN,
        drop_zero=DROP_ZERO_DURATIONS,
        round_decimals=4,
        backend=PLOTTING_BACKEND,
        plot_width=1000,
        plot_height=350,
        show_hover=True,
        show_table=True,
        normalize=DURATION_NORMALIZE,
        float_format=".3f",
    )


## 6. Pitch-class distributions

`display_pitch_class_distributions` shows real (MIDI-derived) and enharmonic (written) pitch-class histograms side by side.


In [ ]:
PC_NORMALIZE = True

if analysis_sources is None:
    print("Skipped. Parse a score first.")
    pc_results = None
else:
    pc_results = display_pitch_class_distributions(
        analysis_sources,
        source_labels=analysis_labels,
        pitch_axis=PITCH_X_AXIS,
        order_axis_by=ORDER_X_AXIS_BY,
        backend=PLOTTING_BACKEND,
        plot_width=1000,
        plot_height=350,
        show_hover=True,
        show_table=True,
        normalize=PC_NORMALIZE,
        float_format=".3f",
    )


## 7. Monophony check (before sequential analyses)

Successive transitions and melodic intervals assume a single note stream. Check overlaps first; with `by_voice=True`, each voice is tested separately.


In [ ]:
# Sequential sections below use this table (selection when present, else full score).
sequential_df = selection if selection is not None and len(selection) else df_pitch
SEQUENTIAL_LABEL = "selection" if sequential_df is selection else "full score"

if sequential_df is None:
    print("Skipped. Parse a score first.")
    mono = None
else:
    mono = check_monophonic_input(
        sequential_df,
        label=SEQUENTIAL_LABEL,
        by_voice=True,
    )
    display(mono)


## 8. Successive pitch transition heatmaps

Bigram heatmaps: rows = previous pitch, columns = next pitch (absolute pitch and pitch class). Computed within each voice, then pooled when `BY_VOICE = True`.


In [ ]:
TRANSITION_NORMALIZE = "count"  # "count" | "row" | "column" | "all"
BY_VOICE = True
REQUIRE_MONOPHONIC = True

if sequential_df is None:
    print("Skipped. Parse a score first.")
    transition_result = None
else:
    transition_result = display_successive_pitch_transition_heatmaps(
        sequential_df,
        source_labels=[SEQUENTIAL_LABEL],
        backend=PLOTTING_BACKEND,
        normalize=TRANSITION_NORMALIZE,
        by_voice=BY_VOICE,
        require_monophonic=REQUIRE_MONOPHONIC,
        plot_width=1000,
        plot_height_pitch=600,
        plot_height_pc=400,
        show_hover=True,
        show_table=True,
        float_format=".3f",
    )


## 9. Melodic interval distribution

Signed interval labels (for example `+M2`, `-P5`) from successive notes. Optional comparison of selection vs full score stays limited to this one parsed file.


In [ ]:
INTERVAL_NORMALIZE = True

if analysis_sources is None:
    print("Skipped. Parse a score first.")
    interval_counts = None
else:
    interval_counts = display_melodic_interval_distribution(
        analysis_sources,
        source_labels=analysis_labels,
        by_voice=True,
        require_monophonic=True,
        normalize=INTERVAL_NORMALIZE,
        backend=PLOTTING_BACKEND,
        plot_width=1000,
        plot_height=350,
        show_hover=True,
        show_table=True,
        float_format=".3f",
    )


## 10. Onset-position histogram

Distribution of note onsets within the measure (quarter-note positions). Needs `df_events` / `results` so pickup and incomplete final measures can be aligned from MEI measure metadata.


In [ ]:
ONSET_BIN_SIZE = 0.25                 # rhythmic grid in quarter lengths
ONSET_NORMALIZE = True
ONSET_EDGE_MEASURE_MODE = "merge_to_regular"  # or "split_by_span"
ONSET_USE_MEASURE_METADATA = True
ONSET_SHOW_MEASURE_DEBUG = False      # set True to print per-measure handling

if df_pitch is None:
    print("Skipped. Parse a score first.")
    onset_position_counts = None
else:
    onset_source = df_pitch if selection is None else (selection, df_pitch)
    onset_labels = ["full score"] if selection is None else ["selection", "full score"]
    onset_position_counts = display_onset_position_histogram(
        onset_source,
        source_labels=onset_labels,
        dfs_by_name=dfs_by_name,
        results=results,
        selection=selection,
        full_df=df_pitch,
        bin_size=ONSET_BIN_SIZE,
        normalize=ONSET_NORMALIZE,
        edge_measure_mode=ONSET_EDGE_MEASURE_MODE,
        use_measure_metadata=ONSET_USE_MEASURE_METADATA,
        backend=PLOTTING_BACKEND,
        plot_width=1000,
        plot_height=350,
        show_hover=True,
        show_table=True,
        show_measure_debug=ONSET_SHOW_MEASURE_DEBUG,
        float_format=".3f",
    )


## What was produced?

- One resolved MEI path and a self-contained parse to `df_pitch` / `df_events` (`parse_enharmonic=True`).
- Optional `selection` from the same score (not a multi-file corpus).
- Count tables and plots for pitch, duration, pitch class, successive transitions, melodic intervals, and onset position.
- A monophony check before the sequential analyses.

Nothing is written to disk by default. Switch `PLOTTING_BACKEND` between `bokeh` and `plt` as needed; re-run plot cells after changing it.

**Next:** [`binary_roundtrip.ipynb`](binary_roundtrip.ipynb) (MEI ↔ table ↔ binary) and [`binary_pattern_search.ipynb`](binary_pattern_search.ipynb). Guide: [Statistics](../docs/guides/statistics.md). Duration column meanings: [`duration_semantics_examples.ipynb`](duration_semantics_examples.ipynb).
